# Student Success Prediction
## Prédiction du statut académique des étudiants

**SAÉ 5.03 — Mise en œuvre d'un processus de Data Mining**
BUT 3 Sciences des Données · Année universitaire 2026–2027 · Travail en binôme

> Statut du notebook : **en cours de réalisation** — les sections marquées **TODO** restent à compléter.
> Ce travail est réalisé dans un cadre pédagogique et ne constitue pas un outil de décision opérationnel.

## Informations sur le projet et les auteurs

| Élément | Valeur |
|---|---|
| Projet | Student Success Prediction — SAÉ 5.03 Data Mining |
| Formation | BUT 3 Sciences des Données |
| Auteurs | Membre 1 · Membre 2 *(TODO : renseigner les noms)* |
| Données | `parcours_etudiants.csv` (fourni sur Moodle, non redistribué) |
| Variable cible | `statut_academique` (Abandon, En cours, Diplômé) |
| Livrables | Ce notebook · Synthèse décisionnelle (`reports/executive_summary.md`) |

## Table des matières

- Contexte et problématique
- Objectifs de l'étude
- Présentation des données
- Imports et configuration
- **Mission 1** — Exploration et audit de la qualité des données
- **Protocole expérimental** — défini avant tout traitement appris (sections P.1 à P.6)
- **Mission 2** — Prétraitement et préparation des données
- **Mission 3** — Construction des modèles de classification
- **Mission 4** — Validation croisée et ajustement des hyperparamètres
- **Mission 5** — Évaluation et interprétation des performances
- **Mission 6** — Interprétation du modèle et recommandations
- Limites de l'étude
- Conclusion générale
- Reproductibilité et références

Les sous-sections reprennent la numérotation officielle des missions (1.1, 2.1, …).

## Contexte et problématique

Un établissement d'enseignement supérieur souhaite mieux comprendre les parcours académiques de ses
étudiants afin d'améliorer ses dispositifs d'accompagnement et d'aide à la réussite.

L'objectif est de **prédire le statut académique d'un étudiant à l'échéance normale de sa formation**,
à partir des informations disponibles à l'inscription et jusqu'à la fin du deuxième semestre.
Il ne s'agit donc pas d'une prédiction au moment de l'inscription.

| Classe | Signification |
|---|---|
| Abandon | L'étudiant a interrompu sa formation. |
| En cours | L'étudiant est encore inscrit à l'échéance normale de la formation. |
| Diplômé | L'étudiant a obtenu son diplôme. |

**Question directrice :** comment construire, à partir de données imparfaites et de classes
déséquilibrées, un modèle de classification fiable, interprétable et évalué selon une démarche
scientifique rigoureuse ?

## Objectifs de l'étude

- Garantir la qualité des données exploitées (audit, nettoyage justifié).
- Construire et comparer une référence élémentaire et quatre algorithmes : KNN, CART, Random Forest, XGBoost.
- Évaluer rigoureusement la capacité de généralisation (protocole sans fuite de données, test utilisé une seule fois).
- Dépasser l'Accuracy : analyser la reconnaissance de chaque classe.
- Interpréter les facteurs associés au parcours académique, sans conclure à une causalité.
- Formuler une recommandation argumentée à destination de l'établissement.

## Présentation des données

- **Source :** export administratif et pédagogique préparé pour l'exercice (fichier fourni sur Moodle).
- **Format :** CSV, séparateur virgule, encodage UTF-8 avec BOM.
- **Unité statistique :** un étudiant, identifié par `id_etudiant` (identifiant administratif fictif).
- **Familles de variables :** identification et inscription, contexte économique, résultats des
  semestres 1 et 2, métadonnées d'export, variable cible.

Le dictionnaire complet est documenté dans `docs/data_dictionary.md`.

> **TODO** — Compléter cette présentation après la mission 1 (dimensions, types, particularités observées).

## Imports et configuration

Les paramètres partagés (chemins, cible, graine, proportion de test, nombre de plis) sont centralisés
dans `src/config.py`. Les chemins sont relatifs à la racine du projet : le notebook s'exécute sur les
deux ordinateurs du binôme sans modification.

In [ ]:
# Localisation de la racine du projet (dossier contenant pyproject.toml et src/),
# pour importer le package src/ quel que soit le dossier d'ouverture du notebook.
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    """Remonte l'arborescence jusqu'à la racine du projet."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Racine du projet introuvable : ouvrez ce notebook depuis le dossier du dépôt."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from importlib.metadata import PackageNotFoundError, version

import pandas as pd

from src import config
from src.data_loader import load_raw_data

pd.set_option("display.max_columns", 50)

In [ ]:
print(f"Racine du projet     : {PROJECT_ROOT.name}")
print(f"Variable cible       : {config.TARGET}")
print(f"Classes attendues    : {', '.join(config.TARGET_CLASSES)}")
print(f"Identifiant étudiant : {config.STUDENT_ID}")
print(f"Graine aléatoire     : {config.RANDOM_STATE}")
print(f"Part du jeu de test  : {config.TEST_SIZE:.0%}")
print(f"Plis de validation   : {config.N_SPLITS}")

### Chargement des données brutes

Le fichier original n'est jamais modifié. Toute transformation s'effectue sur une copie en mémoire.

In [ ]:
df_raw = load_raw_data()
print(f"Données chargées : {df_raw.shape[0]} lignes × {df_raw.shape[1]} colonnes")

## Mission 1 — Exploration et audit de la qualité des données

**Objectifs :** réaliser une analyse approfondie du jeu de données avant toute modélisation,
puis produire un rapport d'audit synthétique.

**Règle :** toute modification des données doit être justifiée (voir `docs/decision_log.md`).

### 1.1 Structure générale du jeu de données

**Question(s) :** Combien d'observations et de variables ? Quels types sont lus ? Le fichier est-il lu correctement ?

In [ ]:
# TODO (1.1)
# - dimensions, types lus par pandas, aperçu de quelques lignes
# - identifier les colonnes attendues numériques mais lues comme texte

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 1.2 Nature et rôle des variables

**Question(s) :** Quelles variables sont quantitatives, qualitatives, ordinales ? Lesquelles sont identifiants, métadonnées, explicatives candidates ou cible ?

In [ ]:
# TODO (1.2)
# - tableau variable / type métier / rôle, confronté au dictionnaire des variables
# - vérifier que chaque variable est disponible au plus tard à la fin du semestre 2

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 1.3 Distributions des variables

**Question(s) :** Quelles formes, asymétries, valeurs extrêmes ou modalités rares observe-t-on ?

In [ ]:
# TODO (1.3)
# - statistiques descriptives et graphiques adaptés (numériques / catégorielles)
# - distinguer erreur de saisie et phénomène réel pour les valeurs atypiques

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 1.4 Répartition des observations entre les classes

**Question(s) :** Les classes sont-elles déséquilibrées ? Quelles conséquences pour l'apprentissage et la métrique ?

In [ ]:
# TODO (1.4)
# - effectifs et proportions de chaque classe de statut_academique
# - vérifier que les modalités observées correspondent aux trois classes attendues

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 1.5 Qualité, cohérence et pertinence des informations

**Question(s) :** Valeurs manquantes, doublons, identifiants répétés, valeurs hors domaine, incohérences entre variables ?

In [ ]:
# TODO (1.5)
# - valeurs manquantes (globales et par classe)
# - doublons exacts et id_etudiant répétés (impact direct sur le protocole de validation)
# - domaines : notes sur 200 et sur 20, entiers non négatifs, modalités Oui/Non, Jour/Soir
# - cohérence entre variables (UE inscrites / validées / créditées / sans évaluation, S1 et S2)

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 1.6 Relations et redondances entre les variables

**Question(s) :** Quelles variables sont fortement liées ou redondantes ? Lesquelles sont liées à la cible ?

In [ ]:
# TODO (1.6)
# - corrélations entre variables numériques, associations entre variables catégorielles
# - redondances signalées par le dictionnaire : note_admission / note_admission_archive, age_inscription / age_inscription_mois, niveau_dossier / note_diplome_anterieur
# - pertinence des métadonnées version_export et code_campagne

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 1.7 Difficultés susceptibles d'affecter les algorithmes

**Question(s) :** Quels problèmes (déséquilibre, échelles, encodage, redondance, fuite potentielle) pèseront sur les modèles ?

In [ ]:
# TODO (1.7)
# - synthèse des risques par algorithme (KNN sensible aux échelles, etc.)

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 1.8 Rapport d'audit synthétique

> **TODO** — Compléter à partir des constats réels des sections 1.1 à 1.7.

| Problème identifié | Variables concernées | Éléments de diagnostic | Conséquences potentielles | Solution envisagée |
|---|---|---|---|---|
| *À compléter* | | | | |

## Protocole expérimental (défini avant tout traitement appris)

La mission 4 impose de définir le protocole et la métrique principale **avant les expérimentations**,
et la séparation apprentissage / test **avant tout traitement nécessitant l'estimation de paramètres**.
Cette section est donc placée avant le prétraitement appris (mission 2), même si sa mise en œuvre
complète (optimisation) est présentée en mission 4.

| Élément | Valeur imposée |
|---|---|
| Séparation | Stratifiée, 80 % apprentissage / 20 % test |
| Graine aléatoire | 42 |
| Validation croisée | Stratifiée, 5 plis, sur le jeu d'apprentissage uniquement |
| Plis | Identiques pour toutes les configurations |
| Étudiants | Aucun étudiant commun entre ensembles, ni entre apprentissage et validation d'un pli |
| Recherche d'hyperparamètres | Boucles explicites, sans `GridSearchCV` |
| Jeu de test | Utilisé une seule fois, pour le modèle final |

### P.1 Choix de la métrique principale

**Question :** quelle métrique traduit le mieux l'objectif (reconnaître chaque statut, y compris
les classes minoritaires) compte tenu du déséquilibre observé en 1.4 ?

Le **F1-score macro** est une candidate envisageable : il donne le même poids à chaque classe.
Le choix définitif doit être justifié ici, **avant l'ajustement des hyperparamètres**, et consigné
dans `docs/decision_log.md`.

> **TODO** — Rédiger la justification de la métrique principale et des métriques secondaires.

### P.2 Identifiants et doublons avant séparation

**Question(s) :** Un même étudiant peut-il apparaître plusieurs fois ? Quelle stratégie garantit qu'il ne figure que dans un seul ensemble ?

In [ ]:
# TODO (P.2)
# - appliquer la décision issue de l'audit 1.5 (dédoublonnage justifié ou découpage groupé stratifié)
# - cette étape doit précéder la séparation apprentissage / test

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### P.3 Séparation stratifiée apprentissage / test

**Question(s) :** Les proportions de classes sont-elles préservées dans les deux ensembles ?

In [ ]:
# TODO (P.3)
# - séparation 80 / 20 stratifiée sur config.TARGET avec random_state=config.RANDOM_STATE
# - mettre le jeu de test de côté : il n'est plus utilisé avant la section 5.3

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### P.4 Définition unique des plis de validation croisée

**Question(s) :** Les 5 plis sont-ils stratifiés et réutilisés à l'identique par toutes les configurations ?

In [ ]:
# TODO (P.4)
# - créer les 5 plis une seule fois (random_state=config.RANDOM_STATE) et stocker leurs indices

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### P.5 Contrôle d'absence de chevauchement d'étudiants

**Question(s) :** Un id_etudiant figure-t-il à la fois en apprentissage et en test, ou en apprentissage et en validation d'un pli ?

In [ ]:
# TODO (P.5)
# - vérifications explicites par assertions (intersection vide des identifiants)

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### P.6 Règles de prévention des fuites de données

- Aucune transformation apprise (imputation, mise à l'échelle, encodage, sélection) n'est ajustée avant la séparation.
- En validation croisée, les transformations sont ajustées sur la partie apprentissage de chaque pli (`Pipeline`).
- Le jeu de test n'intervient ni dans le prétraitement, ni dans la sélection des modèles ou des hyperparamètres.
- Les variables non disponibles à la fin du semestre 2, ou dérivées de la cible, sont exclues.

## Mission 2 — Prétraitement et préparation des données

**Objectifs :** construire une démarche de prétraitement fondée sur l'audit et sur la signification
métier des variables. Chaque décision importante explique **pourquoi** le traitement est nécessaire,
**pourquoi** il a été retenu et **quelles conséquences** il a sur les données.

Le nettoyage déterministe (sans paramètre appris) peut être appliqué de la même manière à tous les
ensembles. Les transformations apprises respectent le protocole expérimental (sections P.1 à P.6).

### 2.1 Stratégie de nettoyage et cohérence des informations

**Question(s) :** Quelles corrections déterministes (types, modalités, incohérences) sont justifiées ?

In [ ]:
# TODO (2.1)
# - appliquer les corrections décidées en 1.8 sur une copie : df = df_raw.copy()
# - documenter chaque correction dans docs/decision_log.md

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 2.2 Variables retenues pour la classification

**Question(s) :** Quelles variables conserver, et pourquoi (sens métier, redondance, disponibilité, risque de fuite) ?

In [ ]:
# TODO (2.2)
# - liste justifiée des variables explicatives ; identifiant et métadonnées traités explicitement

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 2.3 Transformations et représentation des variables

**Question(s) :** Quelles transformations chaque algorithme exige-t-il (encodage, mise à l'échelle, imputation) ?

In [ ]:
# TODO (2.3)
# - construire les ColumnTransformer / Pipeline (ajustés uniquement sur l'apprentissage)

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 2.4 Bilan avant / après prétraitement

**Question(s) :** Qu'est-ce qui a changé (lignes, variables, types, valeurs manquantes) ?

In [ ]:
# TODO (2.4)
# - tableau comparatif avant / après

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

## Mission 3 — Construction des modèles de classification

**Objectifs :** mettre en œuvre la référence élémentaire et les quatre algorithmes en configuration
simple, puis les comparer **sur les mêmes plis de validation** (section P.4).

### 3.1 Principe des algorithmes

> **TODO** — Rédiger pour chaque algorithme son principe, les traitements nécessaires, ses avantages
> et ses limites dans le contexte étudié.

| Algorithme | Principe | Traitements nécessaires | Avantages | Limites |
|---|---|---|---|---|
| Référence (classe majoritaire) | *À compléter* | | | |
| KNN | *À compléter* | | | |
| CART | *À compléter* | | | |
| Random Forest | *À compléter* | | | |
| XGBoost | *À compléter* | | | |

### 3.2 Référence élémentaire

**Question(s) :** Quel score obtient un modèle qui prédit toujours la classe majoritaire du jeu d'apprentissage ?

In [ ]:
# TODO (3.2)
# - DummyClassifier(strategy='most_frequent') évalué sur les plis de P.4

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 3.3 Modèles de référence en configuration simple

**Question(s) :** Quelles performances obtiennent KNN, CART, Random Forest et XGBoost avec des réglages simples ?

In [ ]:
# TODO (3.3)
# - un Pipeline par algorithme ; mêmes plis ; scores par pli conservés

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 3.4 Comparaison des modèles de référence

**Question(s) :** Quel gain chaque algorithme apporte-t-il par rapport à la référence élémentaire ?

In [ ]:
# TODO (3.4)
# - tableau moyenne ± écart-type sur les plis pour la métrique principale et les secondaires

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

## Mission 4 — Validation croisée et ajustement des hyperparamètres

**Objectifs :** ajuster les hyperparamètres par **boucles explicites** (sans `GridSearchCV`), en
utilisant les plis définis en P.4, puis sélectionner le modèle final et le réentraîner sur
l'intégralité du jeu d'apprentissage.

Pour chaque expérimentation : valeurs testées et justification, score moyen, variabilité entre plis,
graphique, interprétation.

### 4.1 KNN — nombre de voisins

**Question(s) :** Comment le score de validation évolue-t-il avec le nombre de voisins ? Où se situe le compromis biais / variance ?

In [ ]:
# TODO (4.1)
# - justifier la grille de valeurs ; boucle explicite ; scores apprentissage et validation par pli

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 4.2 CART — profondeur maximale

**Question(s) :** À partir de quelle profondeur l'arbre sur-apprend-il ?

In [ ]:
# TODO (4.2)
# - justifier la grille ; boucle explicite ; écart apprentissage / validation

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 4.3 Random Forest — nombre d'arbres

**Question(s) :** Le score se stabilise-t-il au-delà d'un certain nombre d'arbres ? Quel coût de calcul ?

In [ ]:
# TODO (4.3)
# - justifier la grille ; boucle explicite ; temps de calcul

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 4.4 XGBoost — nombre d'itérations de boosting

**Question(s) :** Combien d'itérations avant stagnation ou sur-apprentissage ?

In [ ]:
# TODO (4.4)
# - justifier la grille ; boucle explicite ; cible encodée en entiers de façon reproductible

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 4.5 Sélection et réentraînement du modèle final

**Question(s) :** Quel modèle retenir selon la métrique principale, en tenant compte de la variabilité entre plis ?

In [ ]:
# TODO (4.5)
# - sélection argumentée ; réentraînement sur l'intégralité du jeu d'apprentissage

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

## Mission 5 — Évaluation et interprétation des performances

**Objectifs :** comparer Accuracy, Precision, Recall, F1-score (par classe, macro et pondéré) et la
matrice de confusion ; présenter la synthèse de validation croisée des quatre algorithmes puis
l'évaluation finale du modèle retenu sur le jeu de test.

### 5.1 Synthèse des performances en validation croisée

**Question(s) :** Comment se comparent les quatre algorithmes optimisés et la référence élémentaire ?

In [ ]:
# TODO (5.1)
# - tableau de synthèse (moyenne ± écart-type) pour toutes les métriques

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 5.2 Sensibilité du classement à la métrique

**Question(s) :** Le classement des modèles dépend-il de la métrique utilisée ?

In [ ]:
# TODO (5.2)
# - comparer les classements selon Accuracy, F1 macro et F1 pondéré

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 5.3 Évaluation finale sur le jeu de test (une seule fois)

**Question(s) :** Quelles performances le modèle retenu obtient-il sur des données jamais utilisées ?

In [ ]:
# TODO (5.3)
# - évaluation unique du modèle final ; aucune décision ne doit en découler a posteriori

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 5.4 Matrice de confusion et erreurs de classification

**Question(s) :** Quelles classes sont confondues entre elles ? Quelles sont les erreurs principales ?

In [ ]:
# TODO (5.4)
# - matrice de confusion (effectifs et proportions) dans l'ordre config.TARGET_CLASSES

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 5.5 Réponses aux questions de la mission 5

> **TODO** — Répondre à partir des résultats réels.

1. Les différentes classes sont-elles reconnues avec la même efficacité ? Quelles sont les plus difficiles à prédire ?
2. Quelles sont les principales erreurs de classification ?
3. Une Accuracy élevée est-elle suffisante pour considérer le modèle comme performant ?
4. Le classement des modèles dépend-il de la métrique utilisée ?
5. Quelles pourraient être les conséquences des erreurs de classification dans le contexte étudié ?

## Mission 6 — Interprétation du modèle et recommandations

**Objectifs :** comprendre les facteurs associés au parcours académique à partir de l'importance des
variables d'au moins un modèle à arbres, en distinguant **importance prédictive** et **causalité**.

### 6.1 Importance des variables

**Question(s) :** Quelles variables contribuent le plus aux prédictions du modèle à arbres retenu ?

In [ ]:
# TODO (6.1)
# - importance par impureté et/ou par permutation (sur des données non utilisées pour l'entraînement)

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 6.2 Comparaison des importances entre modèles

**Question(s) :** Les modèles à arbres s'appuient-ils sur les mêmes variables ?

In [ ]:
# TODO (6.2)
# - comparer CART, Random Forest et XGBoost si plusieurs modèles sont analysés

> **Interprétation** — *à rédiger après exécution, uniquement à partir des résultats réels.*

### 6.3 Interprétation dans le contexte académique

> **TODO** — Interpréter les importances : une variable importante est **associée** à la prédiction,
> elle n'en est **pas la cause**. Identifier les limites de l'interprétation.

### 6.4 Recommandation à l'établissement

> **TODO** — Recommandation argumentée : modèle retenu, performances, limites et précautions
> nécessaires à son éventuelle utilisation. Le modèle est réalisé dans un cadre pédagogique et ne
> constitue pas un outil de décision opérationnel.

## Limites de l'étude

> **TODO** — Limites liées aux données (qualité, représentativité, variables simulées), au protocole,
> aux modèles et à l'interprétation ; précautions éthiques (risque de stigmatisation, usage
> d'informations socio-économiques, décisions individuelles automatisées).

## Conclusion générale

> **TODO** — Réponse à la question directrice, principaux enseignements, perspectives.

## Reproductibilité et références

**Reproduire l'analyse :**

1. Placer `parcours_etudiants.csv` dans `data/raw/` (voir `data/README.md`).
2. Créer l'environnement : `python -m venv .venv`, puis `python -m pip install -r requirements.txt`.
3. Exécuter le notebook de haut en bas dans un noyau neuf (*Restart Kernel and Run All Cells*).

La graine aléatoire est fixée à `config.RANDOM_STATE` pour toutes les opérations aléatoires.

In [ ]:
print(f"Python {sys.version.split()[0]}")
for package in ("numpy", "pandas", "scikit-learn", "xgboost", "matplotlib", "seaborn"):
    try:
        print(f"{package:<14}{version(package)}")
    except PackageNotFoundError:
        print(f"{package:<14}non installé")

**Références**

- Sujet de la SAÉ 5.03 et dictionnaire des variables (ressources pédagogiques, non redistribuées).
- Documentation scikit-learn : <https://scikit-learn.org/stable/>
- Documentation XGBoost : <https://xgboost.readthedocs.io/>
- Documentation pandas : <https://pandas.pydata.org/docs/>